In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from Bio import SeqIO
import json
import os
import logging
import numpy as np
from scipy.stats import mannwhitneyu

# Import all our custom pipeline modules
from instanexus import preprocessing
from instanexus import assembly
from instanexus import clustering
from instanexus import alignment
from instanexus import consensus
from instanexus import visualization
from instanexus import helpers

# Set up logging to see the pipeline's progress
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

In [ ]:
pd.options.display.max_colwidth = None

In [ ]:
os.chdir('../../../../')

print(f"Current working directory: {os.getcwd()}")

In [ ]:
INPUT_CSV = "_archive/csv/v5/sample_1_preds.csv"

In [ ]:
CONTAMINANTS_PATH = "fasta/contaminants.fasta"

In [ ]:
data = pd.read_csv(INPUT_CSV)

In [ ]:
RUN_NAME = Path(INPUT_CSV).stem

In [ ]:
print(RUN_NAME)

In [ ]:
data.columns

In [ ]:
unique_proteases = data['experiment_name'].str.split('_').str[-3].unique()

print(unique_proteases)

In [ ]:
data["protease"] = data["experiment_name"].apply(
    lambda name: preprocessing.extract_protease(name, unique_proteases)
)

In [ ]:
data.columns

In [ ]:
data["cleaned_preds"] = data["preds"].apply(preprocessing.remove_modifications)

In [ ]:
data["cleaned_preds"][30:50]

In [ ]:
cleaned_psms = data["cleaned_preds"].tolist()

filtered_psms = preprocessing.filter_contaminants(
    cleaned_psms, RUN_NAME , CONTAMINANTS_PATH
)

In [ ]:
data = data[data["cleaned_preds"].isin(filtered_psms)]

In [ ]:
data = data[data['cleaned_preds'].str.len() >= 6]

In [ ]:
data.shape

In [ ]:
data.log_probs

In [ ]:
data_cleaned = preprocessing.clean_instanovo_raw(data)

In [ ]:
data_cleaned.shape

In [ ]:
data_cleaned.columns

In [ ]:
data_cleaned["conf"][:20]

In [ ]:
# number of rows with conf >= 0.9
high_conf_count = (data_cleaned["conf"] >= 0.9).sum()
print(f"Number of high confidence predictions (conf >= 0.9): {high_conf_count}")

In [ ]:
data_cleaned = data_cleaned.dropna(subset=["cleaned_preds"])

In [ ]:
data_cleaned.shape

In [ ]:
# filter dataframe conf >= 0.9
data_filtered = data_cleaned[data_cleaned["conf"] >= 0.9]

In [ ]:
from instanexus.assembly import Assembler

In [ ]:
sequences = data_filtered['cleaned_preds'].tolist()

In [ ]:
assembler = Assembler(
    mode="dbg_weighted",
    kmer_size=6,
    min_overlap=3,
    size_threshold=5,
    min_weight=2
)

In [ ]:
scaffolds = assembler.run(sequences=sequences, df_full=data_filtered)

In [ ]:
len(scaffolds)

In [ ]:
# order in descending order by length
scaffolds = sorted(scaffolds, key=len, reverse=True)

In [ ]:
scaffolds